#　1.ライブラリインポート・定数などの設定

In [3]:
import os
import sys
import json
import sqlite3
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# データパスの設定 (環境に合わせて適宜修正してください)
DB_PATH = "../data/stock_data.db"
TICKER_DICT_PATH = "../data/ticker_dictionary.json"

notebook_path = "/Users/hiranotakahiro/Projects/銘柄スクリーニング検証/notebooks/sector-detec-param.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    data = json.load(f)

modified = False
for cell in data.get("cells", []):
    if cell.get("cell_type") == "code":
        source = cell.get("source", [])
        new_source = []
        for line in source:
            if 'DB_PATH = "../data/stock_data.db"' in line:
                line = line.replace('DB_PATH = "../data/stock_data.db"', 'DB_PATH = "../data/stock_data.db"')
                modified = True
            if 'TICKER_DICT_PATH = "../data/ticker_dictionary.json"' in line:
                line = line.replace('TICKER_DICT_PATH = "../data/ticker_dictionary.json"', 'TICKER_DICT_PATH = "../data/ticker_dictionary.json"')
                modified = True
            new_source.append(line)
        cell["source"] = new_source

if modified:
    with open(notebook_path, "w", encoding="utf-8") as f:
        # 元のフォーマットを崩さないように書き出す
        json.dump(data, f, ensure_ascii=False, indent=1)
    print("Successfully updated notebook paths to reference '../data/' directory.")
else:
    print("No changes made. Target patterns were not found in the notebook.")



# ノートブックでのグラフ表示用設定
%matplotlib inline
plt.rcParams['font.family'] = 'Mainami' # Mac: 'AppleGothic', Windows: 'MS Gothic' など環境に合わせて変更

print("Libraries imported successfully.")


Successfully updated notebook paths to reference '../data/' directory.
Libraries imported successfully.


#　2.マスターデータのロード・データベース接続
ticker_dictionary.jsonからテーマ情報を読み込み、銘柄コードからタグ、タグから銘柄コードを逆引きする辞書を構築。

In [4]:
def load_market_masters(dict_path):
    with open(dict_path, 'r', encoding='utf-8') as f:
        ticker_dict = json.load(f)
    
    # 逆引き用辞書の作成
    code_to_tags = {}
    tag_to_codes = {}
    code_to_name = {}
    
    for item in ticker_dict:
        code = item['code']
        name = item['name']
        tags = item.get('tags', [])
        
        code_to_name[code] = name
        code_to_tags[code] = tags
        
        for tag in tags:
            if tag not in tag_to_codes:
                tag_to_codes[tag] = []
            tag_to_codes[tag].append(code)
            
    print(f"マスター読込完了: 全 {len(code_to_name)} 銘柄, ユニークタグ数 {len(tag_to_codes)}")
    return code_to_name, code_to_tags, tag_to_codes

code_to_name, code_to_tags, tag_to_codes = load_market_masters(TICKER_DICT_PATH)


FileNotFoundError: [Errno 2] No such file or directory: 'data/ticker_dictionary.json'

#　3.株価データ読み込み・テクニカル指標の事前計算
検証高速化のため、過去1年分以上の株価データを一括でPandas DataFrameに読み込み、移動平均出来高、年初来高値などを計算。

In [ ]:
def fetch_and_prepare_data(db_path):
    conn = sqlite3.connect(db_path)
    
    # 【注意】実際のテーブル名、カラム名に合わせてSQLを調整してください
    # ここでは仮に 'daily_prices' というテーブル名、カラム名を想定しています
    query = """
    SELECT date, code, close, volume, market_cap_total AS market_cap 
    FROM daily_prices 
    ORDER BY code, date
    """
    print("データベースから株価データを読み込んでいます...")
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # 銘柄コードを文字列型に統一
    df['code'] = df['code'].astype(str)
    
    # 数値列の型変換（"-"などの文字をNaNに変換）
    for col in ['close', 'volume', 'market_cap']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print("テクニカル指標を計算しています（ローリング処理）...")
    # 5日・25日移動平均出来高の計算
    df['vol_5d_avg'] = df.groupby('code')['volume'].rolling(window=5).mean().reset_index(0, drop=True)
    df['vol_25d_avg'] = df.groupby('code')['volume'].rolling(window=25).mean().reset_index(0, drop=True)
    
    # 年初来高値（または過去250日高値）の計算
    df['high_250d'] = df.groupby('code')['close'].rolling(window=250, min_periods=1).max().reset_index(0, drop=True)
    
    print(f"データ準備完了: 総行数 {len(df)}")
    return df

# 全データをキャッシュとして保持
stock_df_all = fetch_and_prepare_data(DB_PATH)


#　4.検証用コアエンジン
指定したパラメーターに基づき、特定の日付における動意株の抽出とテーマスコアリングを行う関数を定義する。外れ値クリッピングや分母の考慮もオプションでトグル化。

In [ ]:
def analyze_themes_by_date(df_all, target_date, 
                           vol_ratio_threshold=2.0, 
                           price_high_threshold=0.95,
                           clip_outliers=True, 
                           max_vol_ratio=5.0,
                           use_density_filter=True):
    
    # 指定日のデータのみ抽出
    target_date_db = target_date.replace('-', '')
    df_date = df_all[df_all['date'] == target_date_db].copy()
    if df_date.empty:
        print(f"指定された日付 [{target_date}] (検索用: {target_date_db}) のデータが存在しません。市場休業日の可能性があります。")
        return None, None
    
    # Step 1: 動意株のスクリーニング
    df_date['vol_ratio'] = df_date['vol_5d_avg'] / df_date['vol_25d_avg']
    
    # 条件1: 出来高モメンタム
    cond_vol = df_date['vol_ratio'] >= vol_ratio_threshold
    # 条件2: 高値圏ブレイクアウト
    cond_price = df_date['close'] >= (df_date['high_250d'] * price_high_threshold)
    
    # 動意株の確定
    momentum_stocks = df_date[cond_vol & cond_price].copy()
    
    # 外れ値クリッピング（1強ジャック対策）
    if clip_outliers:
        momentum_stocks['processed_vol_ratio'] = momentum_stocks['vol_ratio'].clip(upper=max_vol_ratio)
    else:
        momentum_stocks['processed_vol_ratio'] = momentum_stocks['vol_ratio']
        
    # Step 2: テーマ集計とスコアリング
    theme_scores = {}
    theme_components = {}
    
    for _, row in momentum_stocks.iterrows():
        code = row['code']
        tags = code_to_tags.get(code, [])
        
        # 銘柄ウェイトの計算（時価総額の対数スケーリング）
        # 時価総額が未取得の場合は最低値1として処理
        mcap = row['market_cap'] if pd.notna(row['market_cap']) and row['market_cap'] > 0 else 1
        weight = math.log10(mcap)
        
        # 銘柄単体のスコア貢献度
        stock_score = row['processed_vol_ratio'] * weight
        
        for tag in tags:
            # 「広すぎるタグ」をスキップしたい場合はここに除外リストを追加可能
            if tag in ["電気機器", "電子機器", "円安メリット"]: # 必要に応じて追加
                continue
                
            if tag not in theme_scores:
                theme_scores[tag] = 0.0
                theme_components[tag] = []
                
            theme_scores[tag] += stock_score
            theme_components[tag].append({
                "code": code,
                "name": code_to_name.get(code, "Unknown"),
                "vol_ratio": row['vol_ratio'],
                "market_cap": mcap,
                "stock_score": stock_score
            })
            
    # ランキング用データの成形
    result_records = []
    for tag, score in theme_scores.items():
        components = theme_components[tag]
        num_active_stocks = len(components)
        total_stocks_in_theme = len(tag_to_codes.get(tag, []))
        
        # 活性化率（テーマ内の何％の銘柄が動意づいているか）
        activation_ratio = num_active_stocks / total_stocks_in_theme if total_stocks_in_theme > 0 else 0
        
        # スコアの調整（集団連動を優遇し、分母を考慮するロジック）
        final_score = score
        if use_density_filter:
            # 例: ヒット件数に応じた乗算ボーナス、または活性化率の加算
            final_score = score * (1 + math.log2(num_active_stocks)) * (activation_ratio + 0.5)
            
        # 牽引銘柄の文字列表現を作成
        driving_stocks_str = ", ".join([f"{c['name']}({c['code']})" for c in sorted(components, key=lambda x: x['stock_score'], reverse=True)[:3]])
        
        result_records.append({
            "テーマ名": tag,
            "計算スコア": round(score, 2),
            "最終調整スコア": round(final_score, 2),
            "動意銘柄数": num_active_stocks,
            "テーマ総銘柄数": total_stocks_in_theme,
            "活性化率(%)": round(activation_ratio * 100, 1),
            "牽引銘柄(Top3)": driving_stocks_str,
            "詳細データ": components
        })
        
    df_result = pd.DataFrame(result_records)
    if not df_result.empty:
        df_result = df_result.sort_values(by="最終調整スコア", ascending=False).reset_index(drop=True)
        df_result.insert(0, '順位', df_result.index + 1)
        
    return df_result, momentum_stocks


#　5.検証結果確認

In [ ]:
# ==========================================
# 🔍 検証パラメーター設定エリア
# ==========================================
TARGET_DATE = "2026-04-30"          # 検証したい過去の特定の相場日(YYYY-MM-DD)
VOL_RATIO = 2.0                     # 出来高のベースアップ閾値 (倍)
PRICE_HIGH = 0.95                   # 高値圏維持の閾値 (0.95 = 高値から5%以内)

CLIP_OUTLIERS = True               # モンスター株の1強ジャックを防ぐか
MAX_VOL_RATIO = 5.0                 # 出来高倍率の上限クリッピング値

USE_DENSITY = True                  # テーマ内の密度・活性化率を考慮するか
# ==========================================

# 実行
df_rank, df_stocks = analyze_themes_by_date(
    stock_df_all, 
    target_date=TARGET_DATE,
    vol_ratio_threshold=VOL_RATIO,
    price_high_threshold=PRICE_HIGH,
    clip_outliers=CLIP_OUTLIERS,
    max_vol_ratio=MAX_VOL_RATIO,
    use_density_filter=USE_DENSITY
)

if df_rank is not None and not df_rank.empty:
    print(f"=== {TARGET_DATE} の注目テーマランキング (上位15項目) ===")
    # 画面に綺麗に表示
    display(df_rank[["順位", "テーマ名", "最終調整スコア", "計算スコア", "動意銘柄数", "テーマ総銘柄数", "活性化率(%)", "牽引銘柄(Top3)"]].head(15))
else:
    print("ランクインしたテーマはありませんでした。")


# 6.特定セクターに含まれる銘柄の確認セル

In [ ]:
from IPython.display import display

# 目視確認したいテーマ名を指定
INSPECT_THEME = "セラミックコンデンサー"

if df_rank is not None and not df_rank.empty:
    theme_row = df_rank[df_rank["テーマ名"] == INSPECT_THEME]
    
    if not theme_row.empty:
        print(f"🌲 テーマ【{INSPECT_THEME}】の動意銘柄ディテール:")
        details = theme_row.iloc[0]["詳細データ"]
        df_details = pd.DataFrame(details)
        # スコア貢献度順にソートして表示
        display(df_details.sort_values(by="stock_score", ascending=False).reset_index(drop=True))
        
        # 逆に、そのテーマに属しているのに「今回スクリーニングに漏れた銘柄」をあぶり出す
        all_codes_in_theme = tag_to_codes.get(INSPECT_THEME, [])
        active_codes = [c["code"] for c in details]
        missed_codes = [c for c in all_codes_in_theme if c not in active_codes]
        
        if missed_codes:
            print(f"❌ 条件を満たさず漏れた銘柄 (全 {len(missed_codes)} 銘柄):")
            missed_records = []
            df_date_all = stock_df_all[stock_df_all['date'] == TARGET_DATE.replace('-', )]
            for mc in missed_codes:
                m_stock_data = df_date_all[df_date_all['code'] == mc]
                if not m_stock_data.empty:
                    row = m_stock_data.iloc[0]
                    missed_records.append({
                        "コード": mc,
                        "銘柄名": code_to_name.get(mc, "Unknown"),
                        "出来高倍率": round(row['volume'] / row['vol_25d_avg'], 2) if pd.notna(row['vol_25d_avg']) and row['vol_25d_avg'] > 0 else 0,
                        "高値からの位置(%)": round((row['close'] / row['high_250d']) * 100, 1) if pd.notna(row['high_250d']) and row['high_250d'] > 0 else 0,
                    })
            display(pd.DataFrame(missed_records))
    else:
        print(f"指定されたテーマ '{INSPECT_THEME}' は本日のランキング(圏外含む)に存在しません。")
